# 7.1 线性模型训练

用线性回归把几个特征加权相加来预测需求量：在训练期定权重，在验证期和基线比。
特征组合试了四次（见「运行记录」），这份 notebook 留下的是最后定稿的那次；
FINAL=True 时才用训练+验证期重新拟合，并在最终评估期评一次——那段数据只看这一次。

In [1]:
# 参数：dsflow run 时用 --param FEATURES=... 覆盖；默认就是定稿那次的设置
FEATURES = "lag1,lag2,lag3,mean6,season"
FINAL = True


In [2]:
import json
import sys
from pathlib import Path

import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from demo_lib import FEATURES as FEATURE_NAMES, fit, mae, out, predict

OUT = out("7.1")
FEAT = out("4.1") / "features.parquet"
BASELINE_COLUMN = {"MAE_上月值": "lag1", "MAE_近3月均值": "mean3", "MAE_近6月均值": "mean6"}
features = FEATURES.split(",")

run = dsflow.start_run(
    "7.1", project=ROOT,
    hypothesis=f"线性回归（{'、'.join(FEATURE_NAMES[f] for f in features)}）的验证期 MAE 低于最好的基线")
run.log_input(FEAT, name="features")
feat = pd.read_parquet(FEAT)
train, valid, test = (feat[feat["划分"] == s] for s in ("训练", "验证", "最终评估"))
run.log_params({"特征": features, "模型": "线性回归（最小二乘，预测值截到 0 以上）", "最终模型": FINAL})
print(f"训练 {len(train):,} 行、验证 {len(valid):,} 行、最终评估 {len(test):,} 行；本次特征：{features}")


训练 36,000 行、验证 9,000 行、最终评估 9,000 行；本次特征：['lag1', 'lag2', 'lag3', 'mean6', 'season']


In [3]:
model = fit(train, features)
mv = mae(valid["需求量"], predict(model, valid))
run.log_metrics({"MAE_验证": mv})
for month, d in valid.groupby("月份"):
    run.log_metrics({"MAE_验证": mae(d["需求量"], predict(model, d))}, fold=month)
baselines = json.loads((out("6.1") / "baseline_metrics.json").read_text(encoding="utf-8"))
best = baselines["最好的基线"]
best_mae = baselines["验证期"][best]
conclusion = f"验证期 MAE {mv}，最好的基线（{best.removeprefix('MAE_')}）{best_mae}，{'低' if mv < best_mae else '不低'}于基线"
print(json.dumps({"截距": model["intercept"], "系数": model["coef"]}, ensure_ascii=False, indent=1))
print(conclusion)


{
 "截距": 4.852556,
 "系数": {
  "lag1": -0.266673,
  "lag2": -0.26484,
  "lag3": -0.27153,
  "mean6": 0.347184,
  "season": 0.815466
 }
}
验证期 MAE 6.6481，最好的基线（近6月均值）6.9544，低于基线


In [4]:
if FINAL:
    final = fit(pd.concat([train, valid]), features)
    final["season"] = json.loads((out("4.1") / "season_table.json").read_text(encoding="utf-8"))
    final["trained_on"] = "2025-01 至 2026-03（训练 + 验证期）"
    mt = mae(test["需求量"], predict(final, test))
    # 最终评估期的对照用 6.1 选出的同一个最好的基线，口径一致
    base_name = f"MAE_最终评估_{best.removeprefix('MAE_')}基线"
    mb = mae(test["需求量"], test[BASELINE_COLUMN[best]])
    run.log_metrics({"MAE_最终评估": mt, base_name: mb})
    path = OUT / "model.json"
    path.write_text(json.dumps(final, ensure_ascii=False, indent=1, sort_keys=True), encoding="utf-8")
    (OUT / "final_metrics.json").write_text(json.dumps(
        {"MAE_验证": mv, "MAE_验证_最好的基线": best_mae, "MAE_最终评估": mt, base_name: mb, "最终评估行": len(test)},
        ensure_ascii=False, indent=1), encoding="utf-8")
    run.log_artifact(OUT / "final_metrics.json", purpose="最终模型在验证期与最终评估期的 MAE，及同期最好的基线", kind="table")
    print(f"最终评估期 MAE {mt}，同期{best.removeprefix('MAE_')}基线 {mb}")
else:
    print("本次只比较特征组合，不做最终评估、不登记模型（FINAL=False）")


最终评估期 MAE 6.6127，同期近6月均值基线 6.8679


In [5]:
if FINAL:
    # 误差按近 6 月均值分组：看模型在需求量高低不同的 SKU 上是不是都比基线好
    base_col, base_label = BASELINE_COLUMN[best], best.removeprefix("MAE_")
    scored = test.assign(预测需求量=predict(final, test))
    scored = scored.assign(
        近6月均值分组=pd.cut(scored["mean6"], [0, 5, 10, 20, float("inf")], right=False, labels=["0～5", "5～10", "10～20", "20 以上"]),
        模型误差=(scored["需求量"] - scored["预测需求量"]).abs(),
        基线误差=(scored["需求量"] - scored[base_col]).abs(),
    )
    by_level = (scored.groupby("近6月均值分组", observed=True)
                .agg(SKU月=("需求量", "size"), 模型MAE=("模型误差", "mean"), 基线MAE=("基线误差", "mean"),
                     平均需求量=("需求量", "mean"), 平均预测需求量=("预测需求量", "mean"))
                .rename(columns={"基线MAE": f"{base_label}基线MAE"}).round(4).reset_index())
    by_level.to_csv(OUT / "error_by_level.csv", index=False)
    run.log_artifact(OUT / "error_by_level.csv", purpose=f"最终评估期按近 6 月均值分组的误差：模型与{base_label}基线", kind="table")
    ref = run.log_model(path, "sku_demand_linear", description="SKU 月度需求量线性模型（演示）")
    conclusion += (f"；最终评估期 MAE {mt}（同期{best.removeprefix('MAE_')}基线 {mb}），"
                   f"模型登记为 sku_demand_linear {ref['version']}")
    display(by_level)


,近6月均值分组,SKU月,模型MAE,近6月均值基线MAE,平均需求量,平均预测需求量
0,0～5,2991,5.5774,5.0269,5.5644,5.9523
1,5～10,3675,6.7319,6.7303,7.8063,7.5277
2,10～20,2166,7.6423,9.0023,9.6399,9.8013
3,20 以上,168,9.1621,15.1339,12.1369,13.7452


In [6]:
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


验证期 MAE 6.6481，最好的基线（近6月均值）6.9544，低于基线；最终评估期 MAE 6.6127（同期近6月均值基线 6.8679），模型登记为 sku_demand_linear 65cd04dbf982
